# README-Compatible Adapter Inference Demo

このNotebookは README.md に記載された評価条件に合わせて、Nemotron-3-Nano-30B base model + 外部LoRA adapter を vLLM で読み込み、family ごとに数サンプルだけ推論して出力内容を確認するためのものです。

- 推論エンジン: vLLM
- LoRA 読み込み: LoRARequest
- 推論パラメータ: README.md 記載値に固定
- 目的: 精度計測ではなく raw output と boxed answer の確認

In [1]:
!pip install -U --no-index --find-links /kaggle/input/datasets/luciankucera/vllm-offline-wheels torch vllm datasets

Looking in links: /kaggle/input/datasets/luciankucera/vllm-offline-wheels
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/vllm-0.17.1-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/datasets-4.8.2-py3-none-any.whl
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (from torch)
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl (from torch)
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (from torch)
Processing /kaggle/input/datasets/luciankucera/vllm-offline-wheels/nvidia_cuda_cupti_cu12-12.8.

In [2]:
import polars as pl
import os 
from vllm import LLM, SamplingParams
import torch
from datasets import Dataset
import re
torch.__version__

2026-03-29 13:38:55.765880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774791535.946228      66 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774791535.995206      66 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774791536.480461      66 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774791536.480478      66 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774791536.480479      66 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

'2.10.0+cu128'

In [3]:
import os
import re
from pathlib import Path

import kagglehub
import pandas as pd
import polars as pl

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
ADAPTER_PATH = "/kaggle/input/notebooks/renta0426/infer-rule-based-adapter-readme-compatibl/adapter"
DATASET_CSV = "/kaggle/input/datasets/renta0426/rule-based-training-data/rule_based_verified_600_training_data.csv"
SAMPLES_PER_FAMILY = 10
OTHER_SAMPLES_PER_FAMILY = 4

README_MAX_LORA_RANK = 32
README_MAX_TOKENS = 7680
README_TOP_P = 1.0
README_TEMPERATURE = 0.0
README_MAX_NUM_SEQS = 64
README_GPU_MEMORY_UTILIZATION = 0.85
README_MAX_MODEL_LEN = 8192

adapter_path = Path(ADAPTER_PATH)
required_files = [adapter_path / "adapter_config.json", adapter_path / "adapter_model.safetensors"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Adapter directory must contain adapter_config.json and adapter_model.safetensors. Missing: "
        + ", ".join(missing)
    )

print(f"MODEL_PATH={MODEL_PATH}")
print(f"ADAPTER_PATH={adapter_path}")
print(f"DATASET_CSV={DATASET_CSV}")

MODEL_PATH=/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
ADAPTER_PATH=/kaggle/input/notebooks/renta0426/infer-rule-based-adapter-readme-compatibl/adapter
DATASET_CSV=/kaggle/input/datasets/renta0426/rule-based-training-data/rule_based_verified_600_training_data.csv


In [4]:
CSV_SCHEMA = {
    "id": pl.Utf8,
    "prompt": pl.Utf8,
    "answer": pl.Utf8,
    "generated_cot": pl.Utf8,
    "label": pl.Utf8,
}

frame = pl.read_csv(DATASET_CSV, schema_overrides=CSV_SCHEMA)
family_counts = frame.group_by("label").len().sort("label")
display(family_counts)

binary_frame = (
    frame
    .filter(pl.col("label") == "binary")
    .sort("id")
    .head(SAMPLES_PER_FAMILY)
    .select(["label", "id", "prompt", "answer"])
)

other_frame = (
    frame
    .filter(pl.col("label") != "binary")
    .sort(["label", "id"])
)
other_frame = (
    other_frame
    .group_by("label", maintain_order=True)
    .head(OTHER_SAMPLES_PER_FAMILY)
    .select(["label", "id", "prompt", "answer"])
)

sample_frame = (
    pl.concat([binary_frame, other_frame])
    .sort(["label", "id"])
)

print(
    f"Selected {binary_frame.height} binary prompts and "
    f"{other_frame.height} prompts from other families"
 )
display(sample_frame)

label,len
str,u32
"""binary""",100
"""gravity""",100
"""roman""",100
"""symbol""",100
"""text""",100
"""unit""",100


Selected 10 binary prompts and 20 prompts from other families


label,id,prompt,answer
str,str,str,str
"""binary""","""0031df9c""","""In Alice's Wonderland, a secre…","""00110100"""
"""binary""","""008b52fd""","""In Alice's Wonderland, a secre…","""01100101"""
"""binary""","""030479a6""","""In Alice's Wonderland, a secre…","""01000110"""
"""binary""","""04c44df4""","""In Alice's Wonderland, a secre…","""00000000"""
"""binary""","""06412918""","""In Alice's Wonderland, a secre…","""00010010"""
…,…,…,…
"""text""","""08ad3b0b""","""In Alice's Wonderland, secret …","""the silver cat reads"""
"""unit""","""00208201""","""In Alice's Wonderland, a secre…","""16.65"""
"""unit""","""006a46d3""","""In Alice's Wonderland, a secre…","""19.00"""


In [5]:
def extract_final_answer(text: str | None) -> str:
    if text is None:
        return "NOT_FOUND"

    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [item.strip() for item in matches if item.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        found = re.findall(pattern, text, re.IGNORECASE)
        if found:
            return found[-1].strip()

    numeric = re.findall(r'-?\d+(?:\.\d+)?', text)
    if numeric:
        return numeric[-1]

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def build_prompts(tokenizer, prompt_series: list[str]) -> list[str]:
    prompts = []
    for prompt_text in prompt_series:
        user_content = (
            prompt_text
            + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        )
        try:
            rendered = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            rendered = user_content
        prompts.append(rendered)
    return prompts

In [6]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=README_MAX_NUM_SEQS,
    gpu_memory_utilization=README_GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=README_MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=README_MAX_LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

sampling_params = SamplingParams(
    temperature=README_TEMPERATURE,
    top_p=README_TOP_P,
    max_tokens=README_MAX_TOKENS,
)

tokenizer = llm.get_tokenizer()
prompts = build_prompts(tokenizer, sample_frame['prompt'].to_list())
outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest('adapter', 1, str(adapter_path)),
)

records = []
for row, output, rendered_prompt in zip(sample_frame.iter_rows(named=True), outputs, prompts):
    raw_output = output.outputs[0].text
    records.append({
        'label': row['label'],
        'id': row['id'],
        'expected_answer': row['answer'],
        'extracted_answer': extract_final_answer(raw_output),
        'rendered_prompt': rendered_prompt,
        'raw_output': raw_output,
    })

results_df = pd.DataFrame(records)
print(f"Generated {len(results_df)} samples with README-compatible vLLM settings")

INFO 03-29 13:39:10 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 32, 'enable_chunked_prefill': True, 'model': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-29 13:39:22 [model.py:531] Resolved architecture: NemotronHForCausalLM
INFO 03-29 13:39:22 [model.py:1554] Using max model len 8192
INFO 03-29 13:39:22 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-29 13:39:22 [config.py:618] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
WARNING 03-29 13:39:22 [config.py:381] Mamba cache mode is set to 'all' for NemotronHForCausalLM by default when prefix caching is enabled
INFO 03-29 13:39:22 [config.py:401] Warning: Prefix caching in Mamba cache 'all' mode is currently enabled. Its support for Mamba layers is experimental. Please report any issues you may observe.
INFO 03-29 13:39:22 [config.py:544] Setting attention block size to 2176 tokens to ensure that attention page size is >= mamba page size.
INFO 03-29 13:39:22 [config.py:575] Padding mamba page size by 4.41% to ensure that mamba page size and attention page size are exactly equal.
INFO 03-29 13:39:22 [vllm.py:747] Asynchron

2026-03-29 13:39:27.776303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774791567.786898     463 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774791567.789964     463 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774791567.797555     463 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774791567.797571     463 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774791567.797572     463 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=463) INFO 03-29 13:39:31 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', speculative_config=None, tokenizer='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityCon

...

(EngineCore_DP0 pid=463) INFO 03-29 13:50:36 [gpu_model_runner.py:5386] Graph capturing finished in 55 secs, took -3.62 GiB
(EngineCore_DP0 pid=463) INFO 03-29 13:50:36 [core.py:282] init engine (profile, create kv cache, warmup model) took 75.18 seconds
(EngineCore_DP0 pid=463) INFO 03-29 13:50:37 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-29 13:50:37 [llm.py:388] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/30 [00:00<?, ?it/s]

WARNING 03-29 13:50:38 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generated 30 samples with README-compatible vLLM settings


In [7]:
display(results_df[['label', 'id', 'expected_answer', 'extracted_answer']])

preview_df = results_df.copy()
preview_df['prompt_preview'] = preview_df['rendered_prompt'].str.slice(0, 220)
preview_df['raw_output_preview'] = preview_df['raw_output'].str.slice(0, 600)
display(preview_df[['label', 'id', 'prompt_preview', 'raw_output_preview']])

results_path = '/kaggle/working/rule_based_adapter_readme_inference_samples.csv'
results_df.to_csv(results_path, index=False)
print(f"Saved detailed outputs to {results_path}")

,label,id,expected_answer,extracted_answer
0,binary,0031df9c,00110100,00110100
1,binary,008b52fd,01100101,00011101
2,binary,030479a6,01000110,11000010
3,binary,04c44df4,00000000,00000000
4,binary,06412918,00010010,00010010
5,binary,06698d4e,00110010,00110010
6,binary,069dbaab,00010000,00011100
7,binary,08615ada,11101111,01011110
8,binary,0a195a74,01010010,00110011
9,binary,0c7acd69,11110101,11110101


,label,id,prompt_preview,raw_output_preview
0,binary,0031df9c,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
1,binary,008b52fd,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
2,binary,030479a6,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
3,binary,04c44df4,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to infer the transformation rule. Give...
4,binary,06412918,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
5,binary,06698d4e,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
6,binary,069dbaab,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to infer the transformation rule. Let'...
7,binary,08615ada,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
8,binary,0a195a74,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...
9,binary,0c7acd69,<|im_start|>system\n<|im_end|>\n<|im_start|>us...,We need to find the transformation rule. Let's...


Saved detailed outputs to /kaggle/working/rule_based_adapter_readme_inference_samples.csv


## Notes

- 精度は計算していません。目的は family 横断で出力品質を目視確認することです。
- 期待解は参考表示のみで、評価ロジックには使っていません。
- さらに見る件数を増やす場合は 3番目のセルの SAMPLES_PER_FAMILY を調整してください。
- adapter は外部参照前提なので、2番目のセルの ADAPTER_PATH を実配置に合わせて更新してください。